# Data Integration and Feature Engineering  
## Beatport Tracks + Spotify Audio Features

This notebook builds the first integrated analytical dataset for the project.

Main objectives:

- Inspect the structure of the main relational tables
- Identify the keys available to connect Spotify and Beatport data
- Merge the most relevant tables step by step
- Create derived variables useful for future analysis
- Build a processed dataset for downstream notebooks

In [2]:
# Import core libraries for data manipulation and inspection

import pandas as pd
import numpy as np

# Display all columns when inspecting dataframes
pd.set_option("display.max_columns", None)

In [3]:
# Define project data paths

raw_path = "../data/raw/"
processed_path = "../data/processed/"

In [4]:
# Load the core tables for the integration process
# low_memory=False prevents dtype inference issues when loading large CSV files

audio = pd.read_csv(raw_path + "audio_features.csv", low_memory=False)
sp_track = pd.read_csv(raw_path + "sp_track.csv", low_memory=False)
bp_track = pd.read_csv(raw_path + "bp_track.csv", low_memory=False)

In [5]:
print("audio_features:", audio.shape)
print("sp_track:", sp_track.shape)
print("bp_track:", bp_track.shape)

audio_features: (4687104, 15)
sp_track: (5777707, 10)
bp_track: (10685331, 17)


In [6]:
# Inspect column names of the main tables

print("audio_features columns")
print(audio.columns)

print("\nsp_track columns")
print(sp_track.columns)

print("\nbp_track columns")
print(bp_track.columns)

audio_features columns
Index(['isrc', 'acousticness', 'danceability', 'duration_ms', 'energy',
       'instrumentalness', 'key', 'liveness', 'loudness', 'mode',
       'speechiness', 'tempo', 'time_signature', 'valence', 'updated_on'],
      dtype='object')

sp_track columns
Index(['track_id', 'track_title', 'duration_ms', 'isrc', 'track_number',
       'release_id', 'explicit', 'disc_number', 'preview_url', 'updated_on'],
      dtype='object')

bp_track columns
Index(['track_id', 'title', 'mix', 'is_remixed', 'release_date', 'genre_id',
       'subgenre_id', 'track_url', 'bpm', 'duration', 'duration_ms', 'isrc',
       'key_id', 'label_id', 'release_id', 'updated_on', 'is_matched_spot'],
      dtype='object')


In [7]:
# Check missing values in the ISRC key

print("audio_features missing ISRC:", audio["isrc"].isnull().sum())
print("sp_track missing ISRC:", sp_track["isrc"].isnull().sum())
print("bp_track missing ISRC:", bp_track["isrc"].isnull().sum())

audio_features missing ISRC: 0
sp_track missing ISRC: 46
bp_track missing ISRC: 295778


In [8]:
# Check duplicate ISRC values

print("audio_features duplicated ISRC:", audio["isrc"].duplicated().sum())
print("sp_track duplicated ISRC:", sp_track["isrc"].duplicated().sum())
print("bp_track duplicated ISRC:", bp_track["isrc"].duplicated().sum())

audio_features duplicated ISRC: 0
sp_track duplicated ISRC: 961154
bp_track duplicated ISRC: 3028682


In [9]:
# Merge Spotify tracks with Spotify audio features using ISRC

spotify_tracks = sp_track.merge(
    audio,
    on="isrc",
    how="inner"
)

print("Merged dataset shape:", spotify_tracks.shape)

Merged dataset shape: (5635011, 24)


## Integrating Spotify Track Metadata with Audio Features

The first step in the data integration process consists of combining the
Spotify track metadata with the Spotify audio features.

Both datasets share a common identifier: **ISRC (International Standard Recording Code)**,
which uniquely identifies a musical recording.

Using this identifier, we perform an inner join between:

- `sp_track` (Spotify track metadata)
- `audio_features` (Spotify audio characteristics)

The resulting dataset contains Spotify track information together with
their corresponding musical attributes such as:

- danceability
- energy
- tempo
- valence
- loudness

This integrated dataset represents the foundation for further analysis
and future integration with Beatport metadata.

In [11]:
# Check duplicate keys after merging Spotify tracks and audio features

print("Duplicated ISRC in merged dataset:", spotify_tracks["isrc"].duplicated().sum())
print("Duplicated track_id in merged dataset:", spotify_tracks["track_id"].duplicated().sum())

Duplicated ISRC in merged dataset: 947907
Duplicated track_id in merged dataset: 0


In [12]:
# Inspect structure of the merged Spotify dataset

print(spotify_tracks.columns)
spotify_tracks.head()

Index(['track_id', 'track_title', 'duration_ms_x', 'isrc', 'track_number',
       'release_id', 'explicit', 'disc_number', 'preview_url', 'updated_on_x',
       'acousticness', 'danceability', 'duration_ms_y', 'energy',
       'instrumentalness', 'key', 'liveness', 'loudness', 'mode',
       'speechiness', 'tempo', 'time_signature', 'valence', 'updated_on_y'],
      dtype='object')


,track_id,track_title,duration_ms_x,isrc,track_number,release_id,explicit,disc_number,preview_url,updated_on_x,acousticness,danceability,duration_ms_y,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,updated_on_y
0,05h7hLxcBXnM3dgUZi7YmC,Suckhole,146878,DELJ82099977,23,2MDhYMdgU5GHMj70kc8Ja3,f,1,https://p.scdn.co/mp3-preview/1d7e30dbe20c810a...,2023-08-22 18:08:48,0.200000,0.620,146879,0.585,0.771000,1,0.1690,-15.897,1,0.0356,106,4,0.3210,2023-08-24 09:33:03
1,07E5VE2mJxaEESXkOHIq4J,Terrain,260000,DELJ82099961,7,2MDhYMdgU5GHMj70kc8Ja3,f,1,https://p.scdn.co/mp3-preview/d4ec1529eed983b2...,2023-08-22 18:08:48,0.147000,0.555,260000,0.846,0.311000,9,0.4820,-13.267,1,0.1160,208,4,0.0505,2023-08-24 09:33:03
2,0kg1lYyW3yAGgF7skgn2nx,Groove with You - Live,255666,USSM11501101,8,72MfvP136wxG7aeTUKzyJ7,f,1,https://p.scdn.co/mp3-preview/5f365a24295b2abc...,2023-08-22 18:08:48,0.652000,0.699,255667,0.436,0.000002,11,0.1580,-10.250,0,0.0753,90,4,0.5250,2023-08-28 18:31:07
3,0xvdwCZkGpfA2kJ4rsIW8M,Livin' in the Life / Go for Your Guns - Live,322573,USSM11501105,12,72MfvP136wxG7aeTUKzyJ7,f,1,https://p.scdn.co/mp3-preview/132a1a5867131286...,2023-08-22 18:08:48,0.003330,0.549,322573,0.822,0.005130,10,0.0909,-6.740,0,0.0564,147,4,0.9440,2023-08-28 18:31:07
4,1hTE9UU9I5XThxw1wurbLb,L.g. 1,92000,DELJ82099960,6,2MDhYMdgU5GHMj70kc8Ja3,f,1,https://p.scdn.co/mp3-preview/b4590d953e661c6e...,2023-08-22 18:08:48,0.000028,0.000,92000,1.000,0.934000,6,0.1100,-19.087,1,0.0000,0,0,0.0000,2023-08-24 09:33:03


In [13]:
# Clean duplicated columns generated by the merge

spotify_tracks = spotify_tracks.drop(columns=["duration_ms_y", "updated_on_y"])

spotify_tracks = spotify_tracks.rename(columns={
    "track_id": "spotify_track_id",
    "duration_ms_x": "duration_ms",
    "updated_on_x": "updated_on"
})

print("Dataset shape after cleaning:", spotify_tracks.shape)

spotify_tracks.head()

Dataset shape after cleaning: (5635011, 22)


,spotify_track_id,track_title,duration_ms,isrc,track_number,release_id,explicit,disc_number,preview_url,updated_on,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
0,05h7hLxcBXnM3dgUZi7YmC,Suckhole,146878,DELJ82099977,23,2MDhYMdgU5GHMj70kc8Ja3,f,1,https://p.scdn.co/mp3-preview/1d7e30dbe20c810a...,2023-08-22 18:08:48,0.200000,0.620,0.585,0.771000,1,0.1690,-15.897,1,0.0356,106,4,0.3210
1,07E5VE2mJxaEESXkOHIq4J,Terrain,260000,DELJ82099961,7,2MDhYMdgU5GHMj70kc8Ja3,f,1,https://p.scdn.co/mp3-preview/d4ec1529eed983b2...,2023-08-22 18:08:48,0.147000,0.555,0.846,0.311000,9,0.4820,-13.267,1,0.1160,208,4,0.0505
2,0kg1lYyW3yAGgF7skgn2nx,Groove with You - Live,255666,USSM11501101,8,72MfvP136wxG7aeTUKzyJ7,f,1,https://p.scdn.co/mp3-preview/5f365a24295b2abc...,2023-08-22 18:08:48,0.652000,0.699,0.436,0.000002,11,0.1580,-10.250,0,0.0753,90,4,0.5250
3,0xvdwCZkGpfA2kJ4rsIW8M,Livin' in the Life / Go for Your Guns - Live,322573,USSM11501105,12,72MfvP136wxG7aeTUKzyJ7,f,1,https://p.scdn.co/mp3-preview/132a1a5867131286...,2023-08-22 18:08:48,0.003330,0.549,0.822,0.005130,10,0.0909,-6.740,0,0.0564,147,4,0.9440
4,1hTE9UU9I5XThxw1wurbLb,L.g. 1,92000,DELJ82099960,6,2MDhYMdgU5GHMj70kc8Ja3,f,1,https://p.scdn.co/mp3-preview/b4590d953e661c6e...,2023-08-22 18:08:48,0.000028,0.000,1.000,0.934000,6,0.1100,-19.087,1,0.0000,0,0,0.0000


## Cleaning the Merged Dataset

After merging the Spotify track table with the audio features dataset,
some duplicated columns appear due to overlapping variable names.

In particular:

- `duration_ms_x` and `duration_ms_y`
- `updated_on_x` and `updated_on_y`

These columns originate from both source tables. Since the values are
expected to represent the same information, we keep a single version
and remove the redundant columns.

This step ensures that the dataset remains clean and easier to use
in the following stages of the project.

Additionally, the `track_id` column is renamed to `spotify_track_id`
to make the origin of the identifier explicit.

In [15]:
# Inspect Beatport track table before merging

print("Beatport dataset shape:", bp_track.shape)

print("\nMissing ISRC values:")
print(bp_track["isrc"].isnull().sum())

print("\nDuplicated Beatport track_id:")
print(bp_track["track_id"].duplicated().sum())

Beatport dataset shape: (10685331, 17)

Missing ISRC values:
295778

Duplicated Beatport track_id:
0


In [16]:
# Check duplicated ISRC in Beatport tracks

print("Duplicated ISRC in bp_track:", bp_track["isrc"].duplicated().sum())

Duplicated ISRC in bp_track: 3028682


## Inspecting the Beatport Track Table

Before integrating Beatport metadata with the Spotify dataset, we inspect the
structure and key fields of the `bp_track` table.

Key observations:

- The dataset contains more than **10 million Beatport tracks**.
- The column `track_id` is unique, meaning each row represents a distinct Beatport track.
- A significant number of tracks share the same **ISRC**, which is expected
  because the same recording can appear in multiple releases or versions.
- Some tracks do not contain an ISRC identifier, which prevents them from being
  matched with Spotify data.

Since the integration between Spotify and Beatport relies on the **ISRC**
identifier, we will filter the Beatport dataset to keep only tracks that
contain a valid ISRC before performing the merge.

In [18]:
# Filter Beatport tracks that contain a valid ISRC

bp_track_isrc = bp_track.dropna(subset=["isrc"])

print("Beatport tracks with ISRC:", bp_track_isrc.shape)

Beatport tracks with ISRC: (10389553, 17)


In [19]:
# Check unique ISRC values in Beatport

print("Unique ISRC in Beatport:", bp_track_isrc["isrc"].nunique())

Unique ISRC in Beatport: 7656648


## Beatport ISRC Distribution

The Beatport dataset contains more than 10 million tracks with valid ISRC identifiers.

However, the number of unique ISRC values is significantly smaller. This indicates
that the same recording can appear multiple times in Beatport under different
contexts such as:

- different mixes
- different releases
- compilation appearances
- remastered versions

Because of this one-to-many relationship, merging Beatport data with the Spotify
dataset may increase the number of rows in the final dataset.

Before performing the merge, we will measure how many Spotify tracks can actually
be matched with Beatport tracks using the ISRC identifier.

In [21]:
# Check overlap between Spotify ISRC and Beatport ISRC

spotify_isrc = set(spotify_tracks["isrc"])
beatport_isrc = set(bp_track_isrc["isrc"])

common_isrc = spotify_isrc.intersection(beatport_isrc)

print("Spotify unique ISRC:", len(spotify_isrc))
print("Beatport unique ISRC:", len(beatport_isrc))
print("Common ISRC between datasets:", len(common_isrc))

Spotify unique ISRC: 4687104
Beatport unique ISRC: 7656648
Common ISRC between datasets: 4667811


## Spotify and Beatport Dataset Overlap

To understand how much information can be integrated from Beatport,
we compare the ISRC identifiers present in both datasets.

Results show that almost all Spotify tracks with audio features also
exist in the Beatport dataset.

This high overlap indicates that the integration will allow us to enrich
the Spotify dataset with Beatport metadata such as:

- genre
- subgenre
- label
- BPM
- release date

Because the Beatport dataset may contain multiple tracks with the same
ISRC, the merge may create multiple rows per Spotify track.
This behavior reflects the presence of different mixes and releases
associated with the same recording.

In [23]:
# Select relevant Beatport columns for the integration

bp_selected = bp_track_isrc[
    [
        "track_id",
        "title",
        "genre_id",
        "subgenre_id",
        "bpm",
        "duration_ms",
        "isrc"
    ]
]

print("Beatport selected columns:", bp_selected.shape)

Beatport selected columns: (10389553, 7)


In [24]:
# Merge Spotify tracks with Beatport metadata

tracks_full = spotify_tracks.merge(
    bp_selected,
    on="isrc",
    how="inner"
)

print("Final merged dataset shape:", tracks_full.shape)

Final merged dataset shape: (14460217, 28)


## Integrating Spotify Audio Features with Beatport Metadata

We now merge the Spotify dataset with the Beatport track metadata using the
**ISRC identifier**, which uniquely identifies a recording.

Because Beatport may contain multiple tracks associated with the same ISRC
(for example different mixes or releases), the merge creates a **one-to-many
relationship** between Spotify tracks and Beatport tracks.

As a result, the number of rows increases significantly after the merge.

This behavior is expected and reflects the structure of real music catalog
data, where the same recording may appear multiple times across releases.

In [26]:
# Load Beatport genre tables

bp_genre = pd.read_csv(raw_path + "bp_genre.csv")
bp_subgenre = pd.read_csv(raw_path + "bp_subgenre.csv")

print("Genre table:", bp_genre.shape)
print("Subgenre table:", bp_subgenre.shape)

Genre table: (32, 5)
Subgenre table: (74, 7)


In [27]:
# Inspect genre tables

print(bp_genre.columns)
print(bp_subgenre.columns)

Index(['genre_id', 'genre_name', 'song_count', 'genre_url', 'updated_on'], dtype='object')
Index(['subgenre_id', 'subgenre_name', 'song_count', 'genre_id',
       'subgenre_url', 'updated_on', 'genre_url'],
      dtype='object')


## Adding Beatport Genre Metadata

The Beatport dataset contains additional tables that map genre identifiers
to human-readable genre names.

Two tables are used for this purpose:

- `bp_genre`: maps `genre_id` to the main genre name
- `bp_subgenre`: maps `subgenre_id` to a more specific genre category

By merging these tables with the integrated dataset, we can enrich the data
with descriptive musical categories that will later allow analysis such as:

- distribution of tempo by genre
- energy levels across subgenres
- stylistic differences between electronic music styles

In [29]:
# Select relevant columns from genre tables

genre_selected = bp_genre[
    ["genre_id", "genre_name"]
]

subgenre_selected = bp_subgenre[
    ["subgenre_id", "subgenre_name"]
]

print("Genre selected:", genre_selected.shape)
print("Subgenre selected:", subgenre_selected.shape)

Genre selected: (32, 2)
Subgenre selected: (74, 2)


In [30]:
# Merge genre names into the dataset

tracks_full = tracks_full.merge(
    genre_selected,
    on="genre_id",
    how="left"
)

print("Dataset shape after genre merge:", tracks_full.shape)

Dataset shape after genre merge: (14460217, 29)


In [31]:
# Merge subgenre names into the dataset

tracks_full = tracks_full.merge(
    subgenre_selected,
    on="subgenre_id",
    how="left"
)

print("Dataset shape after subgenre merge:", tracks_full.shape)

Dataset shape after subgenre merge: (14511864, 30)


In [32]:
# Replace missing subgenres with explicit category

tracks_full["subgenre_name"] = tracks_full["subgenre_name"].fillna("Unknown")

tracks_full["subgenre_name"].value_counts().head(10)

subgenre_name
Unknown               13574126
Electro House           413491
Big Room                143228
Future House             74598
Downtempo                63454
Funk / Soul              39840
Uplifting Trance         38141
Ambient                  25031
Glitch Hop               17390
Progressive Trance       15365
Name: count, dtype: int64

In [33]:
# Check missing values in Beatport subgenre identifiers

print("Missing subgenre_id in tracks_full:", tracks_full["subgenre_id"].isna().sum())

Missing subgenre_id in tracks_full: 13574126


In [34]:
# Check missing values after merging subgenre names

print("Missing subgenre_name in tracks_full:", tracks_full["subgenre_name"].isna().sum())

Missing subgenre_name in tracks_full: 0


In [35]:
# Check unique subgenre identifiers

print("Unique subgenre_id in tracks_full:", tracks_full["subgenre_id"].nunique(dropna=True))
print("Unique subgenre_id in bp_subgenre:", bp_subgenre["subgenre_id"].nunique(dropna=True))

Unique subgenre_id in tracks_full: 72
Unique subgenre_id in bp_subgenre: 72


In [36]:
# Detect subgenre IDs that do not match the reference table

unmatched_subgenre_ids = set(tracks_full["subgenre_id"].dropna()) - set(bp_subgenre["subgenre_id"].dropna())

print("Unmatched subgenre_id count:", len(unmatched_subgenre_ids))
print("Example unmatched subgenre_id:", list(unmatched_subgenre_ids)[:10])

Unmatched subgenre_id count: 0
Example unmatched subgenre_id: []


## Genre and Subgenre Integration

Beatport genre metadata was merged into the dataset using the identifiers
`genre_id` and `subgenre_id`.

The main genre table (`bp_genre`) maps each `genre_id` to a unique genre name.

However, the subgenre table may contain multiple rows associated with the
same `subgenre_id`. Because of this, merging the subgenre information slightly
increases the total number of rows in the dataset.

This behavior reflects the hierarchical structure of Beatport's genre
classification system and does not indicate an error in the merge process.

In [38]:
# Convert track duration from milliseconds to minutes

tracks_full["duration_min"] = tracks_full["duration_ms_x"] / 60000

tracks_full["duration_min"].describe()

count    1.451186e+07
mean     5.634832e+00
std      2.265837e+00
min      1.666667e-05
25%      4.260150e+00
50%      5.784000e+00
75%      6.750000e+00
max      1.238770e+02
Name: duration_min, dtype: float64

In [39]:
# Create tempo categories based on BPM ranges common in electronic music

tracks_full["tempo_category"] = pd.cut(
    tracks_full["tempo"],
    bins=[0, 100, 115, 125, 135, 200, 300],
    labels=[
        "slow",
        "downtempo",
        "house_range",
        "techno_range",
        "fast",
        "very_fast"
    ]
)

tracks_full["tempo_category"].value_counts()

tempo_category
house_range     6079319
techno_range    4404372
fast            2001064
slow            1191249
downtempo        805426
very_fast         27892
Name: count, dtype: int64

In [40]:
# Create energy level categories

tracks_full["energy_level"] = pd.cut(
    tracks_full["energy"],
    bins=[0, 0.4, 0.7, 1],
    labels=[
        "low",
        "medium",
        "high"
    ]
)

tracks_full["energy_level"].value_counts()

energy_level
high      8722495
medium    4877464
low        911668
Name: count, dtype: int64

In [85]:
# Replace invalid BPM values

tracks_full["bpm"] = tracks_full["bpm"].replace(0, np.nan)

# Remove duplicated tracks keeping the row with valid BPM

tracks_full = (
    tracks_full
    .sort_values(by="bpm", ascending=False)
    .drop_duplicates(subset="isrc", keep="first")
)

# Reorder dataset by ISRC
tracks_full = tracks_full.sort_values(by="isrc").reset_index(drop=True)

print("Dataset shape after removing duplicates:", tracks_full.shape)

Dataset shape after removing duplicates: (4667811, 33)


## Feature Engineering Insights

Several derived variables were created to enrich the analytical dataset and make it more suitable for exploratory analysis.

During the integration process, multiple rows could appear for the same track because the Beatport catalog may contain several entries associated with the same **ISRC**.  
To avoid duplicated tracks in the analytical dataset, duplicated ISRC values were resolved by prioritizing rows with valid **BPM** information. Rows with `bpm = 0` were treated as missing values.

This step ensures that each row in the final dataset represents a **single unique track**.

### Track duration

The average track duration is approximately **5.6 minutes**, which is consistent with typical electronic music track lengths used in DJ sets.

A small number of extremely long tracks appear in the dataset, which may correspond to extended mixes, live recordings, or potential data anomalies.

### Tempo categories

Most tracks fall within the **115–135 BPM range**, corresponding to the tempo commonly associated with house and techno music.

This confirms the strong presence of club-oriented electronic music in the Beatport catalog.

### Energy levels

The majority of tracks show **high energy values**, reflecting the nature of Beatport's catalog, which focuses heavily on dancefloor-oriented music.

In [88]:
# Select final columns for the analytical dataset

tracks_dataset = tracks_full[
    [
        "spotify_track_id",
        "track_title",
        "isrc",
        "genre_name",
        "subgenre_name",
        "bpm",
        "duration_min",

        # Spotify audio features
        "acousticness",
        "danceability",
        "energy",
        "instrumentalness",
        "key",
        "liveness",
        "loudness",
        "mode",
        "speechiness",
        "tempo",
        "time_signature",
        "valence",

        # Engineered features
        "tempo_category",
        "energy_level"
    ]
]

print("Final dataset shape:", tracks_dataset.shape)
tracks_dataset.head()

Final dataset shape: (4667811, 21)


,spotify_track_id,track_title,isrc,genre_name,subgenre_name,bpm,duration_min,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,tempo_category,energy_level
0,2XRmx1n48eFSEPp9AFuTtF,Bruxism,AAA201604180,Electronica,Unknown,143.0,5.020433,0.000073,0.318,0.690,0.936,9,0.1150,-8.420,1,0.0644,104,4,0.0785,downtempo,medium
1,4BPpQEbXq8rprnSx8AdFKQ,Many,AAA201604181,Electronica,Unknown,120.0,8.544583,0.171000,0.346,0.903,0.122,2,0.5770,-10.111,1,0.6840,193,4,0.0386,fast,high
2,0PjL0BOdLjnRWaEsvehcpG,Onset,AAA201604182,Electronica,Unknown,140.0,7.358950,0.005550,0.349,0.893,0.900,6,0.0625,-8.171,1,0.4270,104,4,0.0349,downtempo,high
3,05GET0OR7Km6yEabSzbOMj,Onset - J. G. Biberkopf Remix,AAA201604183,Electronica,Unknown,86.0,4.200667,0.099800,0.405,0.832,0.847,4,0.0750,-11.891,0,0.1040,171,3,0.0354,fast,high
4,1pzN7qpm3AqSYggL5Pfpl8,Onset - Imaabs Remix,AAA201604184,Electronica,Unknown,83.0,4.626667,0.013400,0.235,0.466,0.799,0,0.0845,-12.491,1,0.0633,88,3,0.0360,slow,medium


In [90]:
# Final dataset inspection

print("Final dataset shape:", tracks_dataset.shape)

tracks_dataset.info()

Final dataset shape: (4667811, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4667811 entries, 0 to 4667810
Data columns (total 21 columns):
 #   Column            Dtype   
---  ------            -----   
 0   spotify_track_id  object  
 1   track_title       object  
 2   isrc              object  
 3   genre_name        object  
 4   subgenre_name     object  
 5   bpm               float64 
 6   duration_min      float64 
 7   acousticness      float64 
 8   danceability      float64 
 9   energy            float64 
 10  instrumentalness  float64 
 11  key               int64   
 12  liveness          float64 
 13  loudness          float64 
 14  mode              int64   
 15  speechiness       float64 
 16  tempo             int64   
 17  time_signature    int64   
 18  valence           float64 
 19  tempo_category    category
 20  energy_level      category
dtypes: category(2), float64(10), int64(4), object(5)
memory usage: 685.5+ MB


In [92]:
tracks_dataset.to_csv(
    processed_path + "electronic_music_tracks_dataset.csv",
    index=False
)